# 02 — Model

Inputs from notebook 01 (corrected **paid-renewal** label), persisted in `data/churn.duckdb`: `pred_points`, `cohorts`, `cohorts_s`.

arc: commitment + logistic floor -> engagement -> xgboost -> scope to paid -> lifecycle -> lock -> significance / split checks -> **model pick on net value** -> full-pop -> calibrate -> cost decision. full rationale in DECISIONS.md.

## setup


In [1]:
import duckdb, pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
con = duckdb.connect('../data/churn.duckdb')
q = lambda sql: con.execute(sql).df()
con.execute("SET enable_progress_bar = false")   # match phase A: quiet duckdb progress widgets


## load phase-A tables
*built + validated in 01 (corrected label) — just confirm they're present*


In [2]:
print(q('''SELECT split, count(*) AS n, count(DISTINCT msno) AS users,
                  round(avg(is_churn::int),4) AS churn_rate
           FROM cohorts_s GROUP BY split ORDER BY min(cohort_month)'''))


   split       n   users  churn_rate
0  train  300000  256907      0.1192
1    val  100000   96706      0.1368
2   test  100000   96314      0.0595


#### tables present
- corrected-label sample loaded: train 11.9% / val 13.7% / test 5.9% churn -> matches the phase A lock
- test base rate ~half of train/val = the real downward drift, carried through on purpose

## commitment features + logistic baseline
- cheapest signal first (the renewal transaction itself) = the PR-AUC floor before any behaviour
- is_cancel held out of features (leakage line); is_free derived from amount


In [3]:
q('''
CREATE OR REPLACE TABLE feat_commitment AS
SELECT c.msno, c.expiry, c.split, c.is_churn,
       t.is_auto_renew, t.payment_plan_days, t.actual_amount_paid, t.plan_list_price,
       (t.actual_amount_paid = 0)::int AS is_free,
       (t.plan_list_price - t.actual_amount_paid) AS discount,
       t.payment_method_id
FROM cohorts_s c
JOIN transactions t
  ON t.msno = c.msno
 AND t.transaction_date = strftime(c.gov_txn,'%Y%m%d')::INT
 AND t.membership_expire_date = strftime(c.expiry,'%Y%m%d')::INT
QUALIFY row_number() OVER (PARTITION BY c.msno, c.expiry ORDER BY t.actual_amount_paid DESC) = 1
''')
print(q('''SELECT is_auto_renew, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM feat_commitment WHERE split='train' GROUP BY is_auto_renew ORDER BY is_auto_renew'''))
print(q('''SELECT is_free, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM feat_commitment WHERE split='train' GROUP BY is_free ORDER BY is_free'''))


   is_auto_renew       n  churn_rate
0              0   52865       0.343
1              1  247135       0.071
   is_free       n  churn_rate
0        0  282884       0.075
1        1   17116       0.845


In [4]:
df = q("SELECT * FROM feat_commitment ORDER BY msno, expiry")
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount']
tr, va = df[df.split=='train'], df[df.split=='val']
ytr, yva = tr['is_churn'].astype(int), va['is_churn'].astype(int)
scaler = StandardScaler().fit(tr[feat_cols])
Xtr, Xva = scaler.transform(tr[feat_cols]), scaler.transform(va[feat_cols])
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
p_va = clf.predict_proba(Xva)[:,1]
print(f"val PR-AUC: {average_precision_score(yva, p_va):.4f}   (no-skill floor = {yva.mean():.4f})")
print(pd.Series(clf.coef_[0], index=feat_cols).sort_values())


val PR-AUC: 0.8586   (no-skill floor = 0.1368)
payment_plan_days    -0.829735
is_auto_renew        -0.624685
discount              0.009590
actual_amount_paid    0.466048
plan_list_price       0.475880
is_free               1.070701
dtype: float64


In [5]:
# interrogate the baseline: is the signal just easy segments? what about look-safe customers?
mask = (va['is_auto_renew']==1) & (va['is_free']==0)
y_seg, p_seg = yva[mask], p_va[mask]
print(f"safe segment: n={mask.sum()}, base={y_seg.mean():.4f}, PR-AUC={average_precision_score(y_seg, p_seg):.4f}")
te = df[df.split=='test']; yte = te['is_churn'].astype(int)
p_te = clf.predict_proba(scaler.transform(te[feat_cols]))[:,1]
print(f"test PR-AUC={average_precision_score(yte, p_te):.4f}  (test base rate={yte.mean():.4f})")


safe segment: n=77645, base=0.0109, PR-AUC=0.0230
test PR-AUC=0.4292  (test base rate=0.0595)


#### commitment floor
- terms alone split the obvious cases: auto_renew=0 churns 34% vs =1 7%; is_free 85% vs paid 7.5%
- logistic floor val PR-AUC 0.859 (no-skill 0.137) -- but driven by the free-trial split (is_free coef +1.07)
- the look-safe slice (auto_renew=1, paid, base 1.1%): PR-AUC 0.023 -> terms can't find these, behaviour next

## engagement features
- commitment reads obvious churns; need behavioural signal for look-safe customers
- recency / activity / completion / trend over a 60d window <= expiry (point-in-time)


In [6]:
q('''
CREATE OR REPLACE TABLE feat_engagement AS
WITH pts AS (
    SELECT msno, expiry, strftime(expiry,'%Y%m%d')::INT AS e_int,
           strftime(expiry-30,'%Y%m%d')::INT AS e_30, strftime(expiry-60,'%Y%m%d')::INT AS e_60
    FROM cohorts_s),
j AS (
    SELECT p.msno, p.expiry, p.e_int, p.e_30, l.date, l.num_25, l.num_50, l.num_75, l.num_985,
           l.num_100, l.num_unq, l.total_secs
    FROM pts p JOIN user_logs l ON l.msno=p.msno AND l.date>p.e_60 AND l.date<=p.e_int),
agg AS (
    SELECT msno, expiry,
           date_diff('day', strptime(max(date)::VARCHAR,'%Y%m%d')::DATE, expiry) AS recency_days,
           count(DISTINCT CASE WHEN date>e_30 THEN date END) AS active_days_30,
           count(DISTINCT CASE WHEN date<=e_30 THEN date END) AS active_days_prior,
           sum(CASE WHEN date>e_30 THEN total_secs ELSE 0 END) AS secs_30,
           sum(CASE WHEN date>e_30 THEN num_unq ELSE 0 END) AS unq_30,
           sum(CASE WHEN date>e_30 THEN num_100 ELSE 0 END) AS completed_30,
           sum(CASE WHEN date>e_30 THEN num_25+num_50+num_75+num_985+num_100 ELSE 0 END) AS plays_30
    FROM j GROUP BY msno, expiry)
SELECT *, completed_30/nullif(plays_30,0) AS completion_ratio,
       active_days_30/nullif(active_days_prior,0) AS activity_trend
FROM agg
''')
print(q("SELECT count(*) AS n_rows, count(DISTINCT msno) AS users FROM feat_engagement"))
print(q('''
SELECT CASE WHEN recency_days<=3 THEN '0-3d' WHEN recency_days<=14 THEN '4-14d'
            WHEN recency_days<=30 THEN '15-30d' ELSE '31-60d' END AS recency_bucket,
       count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
FROM feat_engagement e JOIN cohorts_s c USING (msno, expiry)
WHERE c.split='train' GROUP BY recency_bucket ORDER BY churn_rate
'''))


   n_rows   users
0  393349  317638
  recency_bucket       n  churn_rate
0           0-3d  190240       0.096
1         31-60d    6420       0.140
2          4-14d   32655       0.166
3         15-30d   10612       0.173


In [7]:
q('''
CREATE OR REPLACE TABLE model_data AS
SELECT c.*,
       (e.msno IS NOT NULL)::int AS has_activity_60d,
       coalesce(e.recency_days,60) AS recency_days,
       coalesce(e.active_days_30,0) AS active_days_30,
       coalesce(e.secs_30,0) AS secs_30,
       coalesce(e.unq_30,0) AS unq_30,
       coalesce(e.completion_ratio,0) AS completion_ratio,
       coalesce(e.activity_trend,0) AS activity_trend
FROM feat_commitment c
LEFT JOIN feat_engagement e USING (msno, expiry)
''')
print(q("SELECT count(*) AS n, count(*) FILTER (WHERE has_activity_60d=0) AS silent FROM model_data"))
print(q('''SELECT has_activity_60d, count(*) AS n, round(avg(is_churn::int),3) AS churn_rate
           FROM model_data WHERE split='train' GROUP BY has_activity_60d ORDER BY has_activity_60d'''))


        n  silent
0  500000  106651
   has_activity_60d       n  churn_rate
0                 0   60073       0.157
1                 1  239927       0.110


In [8]:
df = q("SELECT * FROM model_data ORDER BY msno, expiry")
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
tr, va, te = df[df.split=='train'], df[df.split=='val'], df[df.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
scaler = StandardScaler().fit(tr[feat_cols])
Xtr, Xva, Xte = scaler.transform(tr[feat_cols]), scaler.transform(va[feat_cols]), scaler.transform(te[feat_cols])
clf = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
p_va, p_te = clf.predict_proba(Xva)[:,1], clf.predict_proba(Xte)[:,1]
print(f"val  PR-AUC: {average_precision_score(yva,p_va):.4f}")
print(f"test PR-AUC: {average_precision_score(yte,p_te):.4f}")
m = (va.is_auto_renew==1) & (va.is_free==0)
print(f"SAFE segment val: n={m.sum()}, base={yva[m].mean():.4f}, PR-AUC={average_precision_score(yva[m],p_va[m]):.4f}")
print(pd.Series(clf.coef_[0], index=feat_cols).sort_values())


val  PR-AUC: 0.8969
test PR-AUC: 0.5178
SAFE segment val: n=77645, base=0.0109, PR-AUC=0.0329
payment_plan_days    -1.039389
is_auto_renew        -0.689586
active_days_30       -0.344650
unq_30               -0.070834
secs_30               0.017081
discount              0.044822
completion_ratio      0.139653
activity_trend        0.152655
has_activity_60d      0.387529
recency_days          0.439111
actual_amount_paid    0.565547
plan_list_price       0.593437
is_free               1.038552
dtype: float64


In [9]:
# collinearity: read coefs with care
eng   = ['has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
price = ['payment_plan_days','actual_amount_paid','plan_list_price','discount','is_free']
print(tr[eng].corr().round(2)['recency_days'].sort_values())
print()
print(tr[price].corr().round(2))


has_activity_60d   -0.95
completion_ratio   -0.83
active_days_30     -0.72
unq_30             -0.42
activity_trend     -0.23
secs_30             0.02
recency_days        1.00
Name: recency_days, dtype: float64

                    payment_plan_days  actual_amount_paid  plan_list_price  \
payment_plan_days                1.00                0.86             0.97   
actual_amount_paid               0.86                1.00             0.89   
plan_list_price                  0.97                0.89             1.00   
discount                         0.20               -0.26             0.22   
is_free                         -0.10               -0.29            -0.13   

                    discount  is_free  
payment_plan_days       0.20    -0.10  
actual_amount_paid     -0.26    -0.29  
plan_list_price         0.22    -0.13  
discount                1.00     0.34  
is_free                 0.34     1.00  


#### + engagement
- silent (no activity in 60d) churns 15.7% vs active 11.0%; recency leads (0-3d 9.6% rising to ~17% by 15-30d; non-monotonic at the sparse 31-60d bucket)
- logistic +engagement: val 0.897; safe slice 0.033 (up from 0.023) -> small but real lift where it's hard
- features are collinear (recency vs has_activity -0.95; pricing cluster up to 0.97) -> trust importance/direction, not coef magnitudes

## xgboost
- nonlinear + interactions, same 13 feats
- read gain importance with care (collinearity + the is_free root split inflate it)


In [10]:
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','is_free','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
df = q("SELECT * FROM model_data ORDER BY msno, expiry")
tr, va, te = df[df.split=='train'], df[df.split=='val'], df[df.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
clf = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain', tree_method='hist', n_jobs=-1, random_state=42)
clf.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
p_va = clf.predict_proba(va[feat_cols])[:,1]; p_te = clf.predict_proba(te[feat_cols])[:,1]
print(f"trees stopped at: {clf.best_iteration}")
print(f"val  PR-AUC: {average_precision_score(yva,p_va):.4f}")
print(f"test PR-AUC: {average_precision_score(yte,p_te):.4f}")
m = (va.is_auto_renew==1) & (va.is_free==0)
print(f"SAFE segment val: PR-AUC={average_precision_score(yva[m],p_va[m]):.4f}")
print(pd.Series(clf.feature_importances_, index=feat_cols).sort_values(ascending=False))


trees stopped at: 79
val  PR-AUC: 0.9179
test PR-AUC: 0.5842
SAFE segment val: PR-AUC=0.0393
is_free               0.651883
actual_amount_paid    0.174883
is_auto_renew         0.077482
discount              0.047044
active_days_30        0.008909
plan_list_price       0.008071
recency_days          0.008056
activity_trend        0.005946
payment_plan_days     0.005673
unq_30                0.005219
has_activity_60d      0.004337
secs_30               0.001360
completion_ratio      0.001138
dtype: float32


In [11]:
# interrogation: is the topline just a free-trial detector? drop trials and re-score
for name, dfx, y, p in [('val', va, yva, p_va), ('test', te, yte, p_te)]:
    paid = (dfx.is_free == 0)
    print(f"{name} paid-only: n={paid.sum()}, base={y[paid].mean():.4f}, "
          f"PR-AUC={average_precision_score(y[paid], p[paid]):.4f}   (topline {name}: {average_precision_score(y, p):.4f})")


val paid-only: n=89089, base=0.0389, PR-AUC=0.4161   (topline val: 0.9179)
test paid-only: n=98267, base=0.0465, PR-AUC=0.3998   (topline test: 0.5842)


#### xgboost + the free-trial tell
- xgb all-pop: val 0.918 / test 0.586; gain dominated by is_free (0.65, the root split) -> topline is largely a trial detector
- drop trials, re-score paid-only: val 0.416 / test 0.400 -> the real paid task sits at ~0.40, not ~0.9

## scope to paid — DECISION
- model targets paid only (is_free=0); free trials (~93% churn) -> conversion RULE, not a retention offer
- why: cost model assumes a paying customer; trials are trivially separable + have different economics; scoping de-inflates the metric and focuses capacity
- full rationale in DECISIONS.md


In [12]:
feat_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
             'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
dfp = q("SELECT * FROM model_data WHERE is_free = 0 ORDER BY msno, expiry")
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
safe = (va.is_auto_renew == 1)
sc = StandardScaler().fit(tr[feat_cols])
lr = LogisticRegression(max_iter=2000).fit(sc.transform(tr[feat_cols]), ytr)
lr_va, lr_te = lr.predict_proba(sc.transform(va[feat_cols]))[:,1], lr.predict_proba(sc.transform(te[feat_cols]))[:,1]
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain', tree_method='hist', n_jobs=-1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
xg_va, xg_te = xgb.predict_proba(va[feat_cols])[:,1], xgb.predict_proba(te[feat_cols])[:,1]
print(f"paid base rate    : val {yva.mean():.4f}  test {yte.mean():.4f}")
print(f"logistic-paid     : val {average_precision_score(yva,lr_va):.4f}  test {average_precision_score(yte,lr_te):.4f}")
print(f"xgboost-paid      : val {average_precision_score(yva,xg_va):.4f}  test {average_precision_score(yte,xg_te):.4f}")
print(f"xgboost safe slice: val {average_precision_score(yva[safe],xg_va[safe]):.4f}  (n={safe.sum()})")
print(f"trees stopped at  : {xgb.best_iteration}")
print(pd.Series(xgb.feature_importances_, index=feat_cols).sort_values(ascending=False))


paid base rate    : val 0.0389  test 0.0465
logistic-paid     : val 0.3337  test 0.2840
xgboost-paid      : val 0.4167  test 0.4003
xgboost safe slice: val 0.0415  (n=77645)
trees stopped at  : 32
is_auto_renew         0.689675
discount              0.060831
plan_list_price       0.050661
actual_amount_paid    0.045867
payment_plan_days     0.042268
recency_days          0.036959
activity_trend        0.019711
active_days_30        0.017992
unq_30                0.015403
has_activity_60d      0.013491
secs_30               0.003862
completion_ratio      0.003281
dtype: float32


#### paid model (12-feat)
- paid base val 3.9% / test 4.7%; xgb-paid val 0.417 / test 0.400 vs logistic 0.334 / 0.284 -> trees add ~0.08-0.12 over linear
- is_auto_renew dominates gain (0.69); safe slice still ~0.04 -> terms+behaviour only modestly crack the look-safe segment

## lifecycle features
- tenure_days, n_prior_cycles as-of-expiry (point-in-time): does loyalty / history add over renewal + behaviour?


In [13]:
con.execute('''
CREATE OR REPLACE TABLE feat_lifecycle AS
WITH pts AS (SELECT msno, expiry FROM cohorts_s),
     tx  AS (SELECT msno, strptime(transaction_date::VARCHAR,'%Y%m%d')::DATE AS txn_date FROM transactions)
SELECT p.msno, p.expiry,
       date_diff('day', MIN(t.txn_date), p.expiry) AS tenure_days,
       COUNT(DISTINCT t.txn_date)                   AS n_prior_cycles
FROM pts p JOIN tx t ON t.msno = p.msno AND t.txn_date <= p.expiry
GROUP BY p.msno, p.expiry
''')
print(q("SELECT COUNT(*) AS n FROM feat_lifecycle"))
print(q("SELECT MIN(tenure_days) lo, MAX(tenure_days) hi, MIN(n_prior_cycles) clo, MAX(n_prior_cycles) chi FROM feat_lifecycle"))
print(q('''
SELECT q AS tenure_quartile, ROUND(AVG(is_churn::INT),3) AS churn_rate, COUNT(*) AS n FROM (
  SELECT c.is_churn, NTILE(4) OVER (ORDER BY f.tenure_days) AS q
  FROM feat_lifecycle f JOIN cohorts_s c USING (msno, expiry)
) GROUP BY q ORDER BY q
'''))


        n
0  499917
   lo   hi  clo  chi
0   0  789    1   48
   tenure_quartile  churn_rate       n
0                1       0.273  124980
1                2       0.053  124979
2                3       0.068  124979
3                4       0.049  124979


#### lifecycle signal
- newest-tenure quartile churns 27% vs ~5% for older quartiles -> strong first-cycle risk
- tenure 0-789d, n_prior_cycles 1-48 (as-of-expiry, point-in-time)

In [14]:
num_cols = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
            'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend',
            'tenure_days','n_prior_cycles']
cat_cols = ['payment_method_id']; feat_cols = num_cols + cat_cols
dfp = q('''SELECT m.*, f.tenure_days, f.n_prior_cycles FROM model_data m
           LEFT JOIN feat_lifecycle f USING (msno, expiry) WHERE m.is_free = 0 ORDER BY msno, expiry''')
for c in cat_cols: dfp[c] = dfp[c].astype('category')
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)
safe = (va.is_auto_renew == 1)
xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                    eval_metric='aucpr', early_stopping_rounds=50, importance_type='gain',
                    enable_categorical=True, tree_method='hist', n_jobs=-1, random_state=42)
xgb.fit(tr[feat_cols], ytr, eval_set=[(va[feat_cols], yva)], verbose=False)
xg_va, xg_te = xgb.predict_proba(va[feat_cols])[:,1], xgb.predict_proba(te[feat_cols])[:,1]
print(f"paid base          : val {yva.mean():.4f}  test {yte.mean():.4f}")
print(f"xgboost +lifecycle : val {average_precision_score(yva,xg_va):.4f}  test {average_precision_score(yte,xg_te):.4f}")
print(f"safe slice (AR=1)  : val {average_precision_score(yva[safe],xg_va[safe]):.4f}")
print(f"trees stopped at   : {xgb.best_iteration}")
print(pd.Series(xgb.feature_importances_, index=feat_cols).sort_values(ascending=False))


paid base          : val 0.0389  test 0.0465
xgboost +lifecycle : val 0.4758  test 0.4450
safe slice (AR=1)  : val 0.3191
trees stopped at   : 192
payment_method_id     0.223886
n_prior_cycles        0.186005
is_auto_renew         0.174200
recency_days          0.078509
payment_plan_days     0.062793
tenure_days           0.056708
plan_list_price       0.048831
actual_amount_paid    0.033988
activity_trend        0.033731
active_days_30        0.032395
discount              0.027378
unq_30                0.022906
secs_30               0.008081
completion_ratio      0.007359
has_activity_60d      0.003229
dtype: float32


In [15]:
# 1) censoring fingerprint: is tenure bounded by the split (i.e. by calendar)?
print(dfp.groupby('split')['tenure_days'].agg(['mean','max']))

# 2) ablation: add ONE family at a time, watch best_iteration + val/safe
base12 = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
          'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
def run(cols, cats=()):
    for c in cats: dfp[c] = dfp[c].astype('category')
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, enable_categorical=bool(cats),
                      tree_method='hist', n_jobs=-1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    pv = m.predict_proba(va[cols])[:,1]
    return f"iters={m.best_iteration:>4}  val={average_precision_score(yva,pv):.4f}  safe={average_precision_score(yva[safe],pv[safe]):.4f}"

print("base12             :", run(base12))
print("base12 + lifecycle :", run(base12+['tenure_days','n_prior_cycles']))
print("base12 + paymethod :", run(base12+['payment_method_id'], cats=['payment_method_id']))


             mean  max
split                 
test   446.097011  789
train  235.364941  608
val    391.760607  699
base12             : iters=  32  val=0.4167  safe=0.0415
base12 + lifecycle : iters= 172  val=0.4093  safe=0.0843
base12 + paymethod : iters=  99  val=0.4594  safe=0.2794


#### lifecycle / payment_method_id — DECISION (resolved on the corrected label)
- payment_method_id POISONS (high-card categorical, memorised, temporal mix shift) -> DROP
- lifecycle splits two ways: topline -0.0044 [-0.0133,+0.0040] WITHIN noise (no real topline penalty), safe-slice +0.0408 [+0.0292,+0.0530] significantly BETTER (~doubles 0.038 -> 0.079); and the 14-feat overfits (train 0.66 / test 0.40 via the tenure calendar-clock; base12 gap ~0, test even >= train)
- decided on the money, not the AUC: topline is a wash, but in the cost backtest base12 nets Rs 287k vs 257k (12mo) at equal precision (0.50 vs 0.50) and better precision@budget, and it doesn't overfit -> LOCK base12
- the safe-slice gain is real, but a global-threshold contact rule never reaches that low-P segment; it pays off only with segment-specific save rates -> carry lifecycle into phase D, not the scorer
- discovery arc kept: hypothesised lifecycle helps -> topline-neutral but overfits via the tenure clock, sharpens the saveable segment yet doesn't improve the global contact decision -> belongs in the decision layer

In [16]:
# confirm lifecycle on the honest TEST set (val drove the decision; test is reported, not optimised)
safe_te = (te.is_auto_renew == 1)
def run_full(cols):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=-1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    pv, pt = m.predict_proba(va[cols])[:,1], m.predict_proba(te[cols])[:,1]
    return (f"iters={m.best_iteration:>4} | val={average_precision_score(yva,pv):.4f} "
            f"test={average_precision_score(yte,pt):.4f} | "
            f"safe_val={average_precision_score(yva[safe],pv[safe]):.4f} "
            f"safe_test={average_precision_score(yte[safe_te],pt[safe_te]):.4f}")

print("base12             :", run_full(base12))
print("base12 + lifecycle :", run_full(base12 + ['tenure_days','n_prior_cycles']))


base12             : iters=  32 | val=0.4167 test=0.4003 | safe_val=0.0415 safe_test=0.0376
base12 + lifecycle : iters= 172 | val=0.4093 test=0.3959 | safe_val=0.0843 safe_test=0.0785


## locked model
- paid population, **12 features** (base12); payment_method_id and lifecycle both dropped from the scorer (decision above)
- the 14-feat is still fit alongside, only to drive the paired + safe-slice bootstrap that made the call
- store base12 train/val/test preds -> calibrator + honest eval + cost run on these

In [17]:
base12 = ['is_auto_renew','payment_plan_days','actual_amount_paid','plan_list_price','discount',
          'has_activity_60d','recency_days','active_days_30','secs_30','unq_30','completion_ratio','activity_trend']
feat_cols = base12                                      # LOCKED scorer = 12 feats (lifecycle dropped, see decision above)
feat14    = base12 + ['tenure_days','n_prior_cycles']  # 14-feat kept ONLY to test lifecycle's effect
dfp = q('''SELECT m.*, f.tenure_days, f.n_prior_cycles FROM model_data m
           LEFT JOIN feat_lifecycle f USING (msno, expiry) WHERE m.is_free = 0 ORDER BY msno, expiry''')
tr, va, te = dfp[dfp.split=='train'], dfp[dfp.split=='val'], dfp[dfp.split=='test']
ytr, yva, yte = tr.is_churn.astype(int), va.is_churn.astype(int), te.is_churn.astype(int)

def fit_xgb(cols):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=-1, random_state=42)
    m.fit(tr[cols], ytr, eval_set=[(va[cols], yva)], verbose=False)
    return m

xgb     = fit_xgb(feat_cols)                            # the locked 12-feat scorer (we defend this)
xgb_14  = fit_xgb(feat14)                               # reference only -> paired lifecycle test
p_tr = xgb.predict_proba(tr[feat_cols])[:,1]            # train preds -> overfit gap
p_va = xgb.predict_proba(va[feat_cols])[:,1]            # val preds   -> calibrator
p_te = xgb.predict_proba(te[feat_cols])[:,1]            # test preds  -> honest eval + cost
p_te_14 = xgb_14.predict_proba(te[feat14])[:,1]         # 14-feat test -> lifecycle bootstrap only
print(f"locked base12 : test PR-AUC {average_precision_score(yte,p_te):.4f}  "
      f"(14-feat {average_precision_score(yte,p_te_14):.4f})  trees {xgb.best_iteration}")

locked base12 : test PR-AUC 0.4003  (14-feat 0.3959)  trees 32


## how solid is the number?
- every PR-AUC is a point estimate; keep/drop calls (esp. lifecycle, which splits topline vs the safe segment) need a noise check
- bootstrap the test set -> 95% CI; **paired** bootstrap on the difference -> does the lifecycle gain clear noise?
- train-vs-test gap (overfit) and seen-vs-unseen customers (does the recurring-customer split inflate the score?)
- then pick the scorer on the DECISION metric (net value + precision@budget), not the AUC

In [18]:
rng = np.random.default_rng(0)
y, n, B = yte.values, len(yte), 2000
safe_te = (te.is_auto_renew == 1).values                         # look-safe slice = where lifecycle is meant to earn its keep
ap_lock, d_top, d_safe = np.empty(B), np.empty(B), np.empty(B)
for b in range(B):
    idx = rng.integers(0, n, n)                                  # same resampled rows for both models -> paired
    yb  = y[idx]
    ap_lock[b] = average_precision_score(yb, p_te[idx])          # locked base12
    d_top[b]   = average_precision_score(yb, p_te_14[idx]) - ap_lock[b]    # lifecycle effect = 14-feat - base12
    sb = safe_te[idx]                                            # safe slice inside this resample
    d_safe[b]  = (average_precision_score(yb[sb], p_te_14[idx][sb])
                  - average_precision_score(yb[sb], p_te[idx][sb]))

def verdict(d):
    lo, hi = np.percentile(d, [2.5, 97.5])
    tag = 'clears noise (+)' if lo > 0 else 'significantly WORSE (-)' if hi < 0 else 'within noise'
    return lo, hi, tag

lo, hi = np.percentile(ap_lock, [2.5, 97.5])
print(f"locked base12 test PR-AUC: {average_precision_score(y,p_te):.4f}  95% CI [{lo:.4f}, {hi:.4f}]")
for name, d in [('topline   ', d_top), ('safe-slice', d_safe)]:
    dlo, dhi, tag = verdict(d)
    print(f"lifecycle (14-feat) gain {name}: {d.mean():+.4f}  95% CI [{dlo:+.4f}, {dhi:+.4f}]  -> {tag}")

locked base12 test PR-AUC: 0.4003  95% CI [0.3846, 0.4160]
lifecycle (14-feat) gain topline   : -0.0044  95% CI [-0.0133, +0.0040]  -> within noise
lifecycle (14-feat) gain safe-slice: +0.0408  95% CI [+0.0292, +0.0530]  -> clears noise (+)


#### significance — resolved
- lifecycle (14-feat vs base12), paired on the same resamples: topline -0.0044 [-0.0133,+0.0040] WITHIN noise (no real topline penalty); safe-slice +0.0408 [+0.0292,+0.0530] significantly BETTER (~doubles the look-safe slice)
- the safe-slice gain is real; topline is a wash -- so the question is which metric the system acts on, and the 14-feat's train/test overfit (0.66 / 0.40) settles it against using it as the scorer
- the cost rule thresholds calibrated P(churn) GLOBALLY -> the contact list rides the topline ranking, where the two tie and base12 wins on net value (below) without overfitting; the safe-slice gain is parked for the phase-D segment layer
- locked base12 test PR-AUC 0.400, 95% CI [0.385, 0.416]

In [19]:
# overfit gap: train vs test PR-AUC. NOTE train base rate != test base rate, so part of any gap is base-rate, not memorising
for name, m, cols in [('base12 (locked)', xgb, feat_cols), ('14-feat', xgb_14, feat14)]:
    ap_tr = average_precision_score(ytr, m.predict_proba(tr[cols])[:,1])
    ap_te = average_precision_score(yte, m.predict_proba(te[cols])[:,1])
    print(f"{name:16s} train AP {ap_tr:.4f} (base {ytr.mean():.3f})   test AP {ap_te:.4f} (base {yte.mean():.3f})")

base12 (locked)  train AP 0.3551 (base 0.075)   test AP 0.4003 (base 0.046)
14-feat          train AP 0.6631 (base 0.075)   test AP 0.3959 (base 0.046)


In [20]:
# split-design test: does the model do BETTER on customers it saw in train? if not, recurrence isn't inflating the score
train_ids = set(q("SELECT DISTINCT msno FROM cohorts WHERE split='train'")['msno'])
seen = te['msno'].isin(train_ids).values
for name, mk in [('seen in train', seen), ('new (unseen)', ~seen)]:
    yy, pp = yte.values[mk], p_te[mk]
    print(f"{name:14s} n={mk.sum():>6}  base={yy.mean():.4f}  PR-AUC={average_precision_score(yy,pp):.4f}")
# caveat: unseen = newer customers (first seen in test window), so a gap could be new-vs-tenured, not leakage


seen in train  n= 80933  base=0.0362  PR-AUC=0.3254
new (unseen)   n= 17334  base=0.0944  PR-AUC=0.5387


In [21]:
# MODEL SELECTION on the decision, not the AUC: calibrate each candidate, run the cost rule, compare net value.
# also precision@budget at fixed contact volumes -- the realistic operating point (finite retention capacity).
offer, save, horizon = 150, 0.30, 12
pm    = te.payment_plan_days > 0
arpu  = (te.loc[pm,'actual_amount_paid'] / te.loc[pm,'payment_plan_days'] * 30).median()
value = arpu * horizon
be    = offer / (save * value)
yv    = yte.values

def decide(p_va_m, p_te_m):
    cal  = IsotonicRegression(out_of_bounds='clip').fit(p_va_m, yva).predict(p_te_m)  # per-model calibration on val
    mask = cal >= be
    n, tp = int(mask.sum()), int(yv[mask].sum())
    row = {'contacted': n, 'precision': round(tp/n, 3) if n else 0,
           'would_stay': n - tp, 'net_Rs': round(tp*save*value - n*offer)}
    order = np.argsort(-cal)                                                          # rank by calibrated P(churn)
    for k in (1000, 3000):                                                            # precision at a fixed budget
        row[f'prec@{k}'] = round(yv[order[:k]].mean(), 3)
    return row

p_va_14 = xgb_14.predict_proba(va[feat14])[:,1]                                       # 14-feat val preds, for its own calibration
res = pd.DataFrame({'base12 (locked)': decide(p_va, p_te),
                    '14-feat':         decide(p_va_14, p_te_14)}).T
print(f"ARPU Rs {arpu:.0f} | value(12mo) Rs {value:.0f} | break-even {be:.3f}\n")
print(res.to_string())

ARPU Rs 129 | value(12mo) Rs 1548 | break-even 0.323

                 contacted  precision  would_stay    net_Rs  prec@1000  prec@3000
base12 (locked)     3491.0      0.500      1746.0  286728.0      0.698      0.524
14-feat             3110.0      0.501      1552.0  257035.0      0.678      0.506


#### model pick — on the decision, not the AUC
- calibrate each candidate, run the SAME cost rule, compare net value + precision@budget (finite retention capacity)
- base12 wins on the money: net Rs 287k vs 257k (12mo) at equal precision (0.50 vs 0.50), and on precision@budget -- @1k 0.70 vs 0.68, @3k 0.52 vs 0.51 (it contacts a bit more, 3,491 vs 3,110, and still nets more)
- so the lock is decided in rupees + precision@budget, not on the 0.400-vs-0.396 PR-AUC gap -> base12 confirmed

## full-population confirmation
- everything above is on the 300k/100k sample; confirm the locked model holds on the FULL paid population once
- **SLOW**: rebuilds the three feature tables at full scale (the engagement join hits the full log table)


In [22]:
con.execute("SET preserve_insertion_order = false")   # stops order-buffering during large aggregates — the main spill source
con.execute("SET threads = 4")                         # fewer concurrent spill buffers -> lower peak temp

In [23]:
# full-scale: paid commitment -> engagement -> lifecycle, then fit the locked feats and eval on the full test
con.execute('''
CREATE OR REPLACE TABLE fc_full AS
SELECT * FROM (
  SELECT c.msno, c.expiry, c.split, c.is_churn,
         t.is_auto_renew, t.payment_plan_days, t.actual_amount_paid, t.plan_list_price,
         (t.plan_list_price - t.actual_amount_paid) AS discount
  FROM cohorts c
  JOIN transactions t ON t.msno=c.msno
   AND t.transaction_date=strftime(c.gov_txn,'%Y%m%d')::INT
   AND t.membership_expire_date=strftime(c.expiry,'%Y%m%d')::INT
  QUALIFY row_number() OVER (PARTITION BY c.msno, c.expiry ORDER BY t.actual_amount_paid DESC)=1
) WHERE actual_amount_paid > 0
''')

# fe_full: count(*) FILTER instead of count(DISTINCT date) -- valid because user_logs is one row per (msno,date)
con.execute('''
CREATE OR REPLACE TABLE fe_full AS
WITH pts AS (SELECT msno, expiry, strftime(expiry,'%Y%m%d')::INT e_int,
                    strftime(expiry-30,'%Y%m%d')::INT e_30, strftime(expiry-60,'%Y%m%d')::INT e_60 FROM fc_full),
j AS (SELECT p.msno,p.expiry,p.e_int,p.e_30,l.date,l.num_25,l.num_50,l.num_75,l.num_985,l.num_100,l.num_unq,l.total_secs
      FROM pts p JOIN user_logs l ON l.msno=p.msno AND l.date>p.e_60 AND l.date<=p.e_int),
agg AS (SELECT msno, expiry,
               date_diff('day', strptime(max(date)::VARCHAR,'%Y%m%d')::DATE, expiry) recency_days,
               count(*) FILTER (WHERE date>e_30)  active_days_30,
               count(*) FILTER (WHERE date<=e_30) active_days_prior,
               sum(CASE WHEN date>e_30 THEN total_secs ELSE 0 END) secs_30,
               sum(CASE WHEN date>e_30 THEN num_unq ELSE 0 END) unq_30,
               sum(CASE WHEN date>e_30 THEN num_100 ELSE 0 END) completed_30,
               sum(CASE WHEN date>e_30 THEN num_25+num_50+num_75+num_985+num_100 ELSE 0 END) plays_30
        FROM j GROUP BY msno, expiry)
SELECT *, completed_30/nullif(plays_30,0) completion_ratio,
          active_days_30/nullif(active_days_prior,0) activity_trend
FROM agg
''')

# fl_full: pre-distinct transactions to (msno, txn_date), then count(*) == distinct txn dates
con.execute('''
CREATE OR REPLACE TABLE fl_full AS
WITH pts AS (SELECT msno, expiry FROM fc_full),
     tx  AS (SELECT DISTINCT msno, strptime(transaction_date::VARCHAR,'%Y%m%d')::DATE txn_date FROM transactions)
SELECT p.msno, p.expiry,
       date_diff('day', min(t.txn_date), p.expiry) tenure_days,
       count(*) n_prior_cycles
FROM pts p JOIN tx t ON t.msno=p.msno AND t.txn_date<=p.expiry
GROUP BY p.msno, p.expiry
''')

full = q('''
SELECT c.split, c.is_churn, c.is_auto_renew, c.payment_plan_days, c.actual_amount_paid, c.plan_list_price, c.discount,
       (e.msno IS NOT NULL)::int has_activity_60d,
       coalesce(e.recency_days,60) recency_days, coalesce(e.active_days_30,0) active_days_30,
       coalesce(e.secs_30,0) secs_30, coalesce(e.unq_30,0) unq_30,
       coalesce(e.completion_ratio,0) completion_ratio, coalesce(e.activity_trend,0) activity_trend,
       f.tenure_days, f.n_prior_cycles
FROM fc_full c LEFT JOIN fe_full e USING (msno, expiry) LEFT JOIN fl_full f USING (msno, expiry)
''')
trf, vaf, tef = full[full.split=='train'], full[full.split=='val'], full[full.split=='test']
ytrf, yvaf, ytef = trf.is_churn.astype(int), vaf.is_churn.astype(int), tef.is_churn.astype(int)
mf = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8,
                   eval_metric='aucpr', early_stopping_rounds=50, tree_method='hist', n_jobs=-1, random_state=42)
mf.fit(trf[feat_cols], ytrf, eval_set=[(vaf[feat_cols], yvaf)], verbose=False)   # feat_cols = the locked candidate
p_tef = mf.predict_proba(tef[feat_cols])[:,1]
print(f"FULL paid: train {len(trf):,}  val {len(vaf):,}  test {len(tef):,}  (sample was 300k/100k)")
print(f"full-pop test PR-AUC: {average_precision_score(ytef,p_tef):.4f}  base {ytef.mean():.4f}  trees {mf.best_iteration}")

FULL paid: train 9,399,666  val 2,202,951  test 2,317,271  (sample was 300k/100k)
full-pop test PR-AUC: 0.4129  base 0.0465  trees 92


#### full-population confirmation
- locked base12 refit on the FULL paid population (train 9.4M / val 2.2M / test 2.3M): test PR-AUC 0.413 at base 4.7% (trees 92)
- matches the 300k-sample 0.400 (a touch higher) -> the subsample wasn't flattering the model; it holds at scale

## calibration
- the cost rule plugs P(churn) into money -> probabilities must be calibrated
- fit isotonic on val, evaluate on test; PR-AUC (ranking) vs Brier (calibration); reliability by decile


In [24]:
# RAW reliability on test: does predicted prob match observed churn, bin by bin?
chk = pd.DataFrame({'p': p_te, 'y': yte.values})
chk['bin'] = pd.qcut(chk['p'], 10, duplicates='drop')
rel = chk.groupby('bin', observed=True).agg(pred=('p','mean'), actual=('y','mean'), n=('y','size'))
print(rel.round(4))
print()
print(f"max raw predicted prob on test: {p_te.max():.4f}")


                    pred  actual      n
bin                                    
(0.0224, 0.0239]  0.0237  0.0140  13097
(0.0239, 0.0244]  0.0242  0.0138   9806
(0.0244, 0.0278]  0.0273  0.0169  21304
(0.0278, 0.0398]  0.0314  0.0209   4927
(0.0398, 0.0544]  0.0528  0.0172  19469
(0.0544, 0.0548]  0.0548  0.0196   3515
(0.0548, 0.0608]  0.0576  0.0198   6508
(0.0608, 0.0976]  0.0748  0.0470   9826
(0.0976, 0.829]   0.2493  0.2846   9815

max raw predicted prob on test: 0.8291


In [25]:
iso = IsotonicRegression(out_of_bounds='clip').fit(p_va, yva)    # learn score->true-prob on val
p_te_cal = iso.predict(p_te)

chk = pd.DataFrame({'p': p_te_cal, 'y': yte.values})
chk['bin'] = pd.qcut(chk['p'], 10, duplicates='drop')
print(chk.groupby('bin', observed=True).agg(pred=('p','mean'), actual=('y','mean'), n=('y','size')).round(4))
print()
print(f"PR-AUC raw {average_precision_score(yte,p_te):.4f} -> cal {average_precision_score(yte,p_te_cal):.4f}  (should ~match)")
print(f"Brier   raw {brier_score_loss(yte,p_te):.5f} -> cal {brier_score_loss(yte,p_te_cal):.5f}  (lower=better)")
print(f"calibrated max prob: {p_te_cal.max():.4f}")
for thr in [0.63, 0.30, 0.20, 0.10]:
    nn = (p_te_cal >= thr).sum()
    print(f"  P>={thr:.2f}: {nn:>5} customers ({nn/len(p_te_cal)*100:5.2f}% of test)")


                       pred  actual      n
bin                                       
(-0.001, 0.000282]   0.0002  0.0149  15541
(0.000282, 0.00106]  0.0011  0.0157  30475
(0.00106, 0.0138]    0.0104  0.0218   3298
(0.0138, 0.0162]     0.0162  0.0176  23337
(0.0162, 0.0257]     0.0214  0.0208   7867
(0.0257, 0.0796]     0.0558  0.0543   8693
(0.0796, 1.0]        0.3048  0.3026   9056

PR-AUC raw 0.4003 -> cal 0.3880  (should ~match)
Brier   raw 0.03516 -> cal 0.03444  (lower=better)
calibrated max prob: 1.0000
  P>=0.63:   683 customers ( 0.70% of test)
  P>=0.30:  3498 customers ( 3.56% of test)
  P>=0.20:  5418 customers ( 5.51% of test)
  P>=0.10:  8490 customers ( 8.64% of test)


In [26]:
# drift check: the calibrator only transfers as well as val represents the test period
p_va_cal = iso.predict(p_va)
print(f"VAL : mean cal prob {p_va_cal.mean():.4f} vs actual {yva.mean():.4f}  (calibrator's own set -> should match)")
print(f"TEST: mean cal prob {p_te_cal.mean():.4f} vs actual {yte.mean():.4f}  (under-predicts if test churns more)")


VAL : mean cal prob 0.0389 vs actual 0.0389  (calibrator's own set -> should match)
TEST: mean cal prob 0.0393 vs actual 0.0465  (under-predicts if test churns more)


#### calibration
- raw scores rank well but are mis-scaled; isotonic-on-val keeps ranking near-identical (PR-AUC 0.400 -> 0.388) and nudges Brier down (0.0352 -> 0.0344)
- reliability lines up after calibration (top decile predicted 0.305 vs observed 0.303)
- drift: val mean cal prob 0.039 = its own actual; test 0.039 vs actual 0.047 -> mild under-prediction (test churns a bit more) -> watch label shift before trusting absolute probabilities

## decision layer
- contact iff `P(churn) * save * value > offer`; value = data-derived ARPU * horizon
- backtest on REAL test outcomes; the offer is charged to EVERY contact, and we surface the budget spent on would-stay customers (cannibalisation) + a pessimistic net that also nets the discount out of saved revenue


In [27]:
offer, save = 150, 0.30
paid_mask = te.payment_plan_days > 0
arpu = (te.loc[paid_mask,'actual_amount_paid'] / te.loc[paid_mask,'payment_plan_days'] * 30).median()
print(f"data-derived monthly ARPU (paid, median): Rs {arpu:.0f}")
print()
rows = []
for horizon in [6, 12, 18, 24]:
    value = arpu * horizon
    be    = offer / (save * value)                     # break-even prob at this value
    mask  = p_te_cal >= be
    n     = int(mask.sum())
    tp    = int(yte.values[mask].sum())                # actual churners contacted
    fp    = n - tp                                     # would-stay customers contacted (needless discount)
    precision = tp / n if n else 0
    net      = tp * save * value - n * offer                       # offer charged to ALL contacts
    net_pess = tp * save * (value - offer) - n * offer             # + discount eats into saved revenue
    rows.append({'horizon_mo':horizon, 'value_Rs':round(value), 'break_even':round(be,3),
                 'contacted':n, 'precision':round(precision,3),
                 'would_stay_fp':fp, 'wasted_Rs':round(fp*offer),
                 'net_Rs':round(net), 'net_pessimistic_Rs':round(net_pess)})
print(pd.DataFrame(rows).to_string(index=False))


data-derived monthly ARPU (paid, median): Rs 129

 horizon_mo  value_Rs  break_even  contacted  precision  would_stay_fp  wasted_Rs  net_Rs  net_pessimistic_Rs
          6       774       0.646        555      0.832             93      13950   24026                3236
         12      1548       0.323       3491      0.500           1746     261900  286728              208203
         18      2322       0.215       5418      0.412           3186     477900  742111              641671
         24      3096       0.161       8352      0.319           5691     853650 1218737             1098992


#### cost / decision layer
- data-derived monthly ARPU (paid, median) = Rs 129; contact iff calibrated P(churn) > offer/(save*value), value = ARPU*horizon
- gross net is positive at every horizon (6mo Rs 24k -> 24mo Rs 1.22M); the pessimistic net (discount also netted out of saved revenue) is positive at every horizon too (6mo +Rs 3.2k, the thinnest point)
- longer assumed value lowers the bar -> contact more (555 -> 8,352), precision falls (0.83 -> 0.32), more budget wasted on would-stay (cannibalisation surfaced, not hidden)
- 12mo planning anchor: ~3,491 contacted at 0.50 precision, net Rs 287k

In [28]:
# sensitivity to the softest assumption: hold horizon=12mo, sweep the save rate
value = arpu * 12
print(f"value = Rs {value:.0f} (ARPU x 12mo)")
print()
rows = []
for save in [0.15, 0.20, 0.30, 0.40]:
    be   = offer / (save * value)
    mask = p_te_cal >= be
    n    = int(mask.sum()); tp = int(yte.values[mask].sum())
    net  = tp * save * value - n * offer
    rows.append({'save_rate':save, 'break_even':round(be,3), 'contacted':n,
                 'precision':round(tp/n,3) if n else 0, 'net_Rs':round(net)})
print(pd.DataFrame(rows).to_string(index=False))


value = Rs 1548 (ARPU x 12mo)

 save_rate  break_even  contacted  precision  net_Rs
      0.15       0.646        555      0.832   24026
      0.20       0.484       1342      0.651   68981
      0.30       0.323       3491      0.500  286728
      0.40       0.242       4050      0.472  577030


#### sensitivity to save_rate (the softest assumption)
- hold horizon 12mo, sweep save 0.15-0.40: the rule stays net-positive even at save=0.15 -> robust to the guess
- phase D (uplift) measures save instead of assuming it; until then 0.30 is the planning anchor and the sign doesn't flip across the range

## decisions to preempt
- **no hyperparameter search**: fixed sensible defaults + early stopping; at this scale tuning wouldn't beat the noise band, so it wasn't worth the complexity
- **no imbalance reweighting**: deliberately no `scale_pos_weight` -> it distorts the probabilities the cost layer needs calibrated
- **collinear features kept**: trees are robust to the ~0.97 pricing cluster (one gets picked); I'd prune only for a leaner / more-explainable model, not for PR-AUC
- **save_rate is an assumption, not measured**: 0.30 drives the money; Phase D (uplift) grounds it; rupee figures are sample-scale
- PR-AUC is not comparable across splits with different base rates -> compare lift over base, use val for decisions / test for honest reporting
